# S40in × QUN — Kaggle GPU Workbench

Бесплатная проверка GPU и первый реальный тест генерации изображения. Запускай в Kaggle Notebooks с **Accelerator → GPU** и **Internet → On**.

## Что делает ноутбук
1. Проверяет GPU, CUDA, память и доступ к интернету.
2. Устанавливает только необходимые Python-пакеты.
3. Загружает `stabilityai/sd-turbo` и генерирует тестовый рекламный визуал.
4. Сохраняет PNG, метаданные и ZIP в `/kaggle/working/qun_gpu_results/`.

**Важно:** Kaggle — временный пакетный GPU-узел, не постоянный сервер и не гарантированный публичный API. Не добавляй сюда секреты QUN/Supabase и не запускай публичный сетевой туннель. Перед коммерческим использованием проверь актуальную лицензию модели. Не запускай GPU-сессию без необходимости; скачай ZIP до завершения сессии.


In [ ]:
from pathlib import Path
import os, json, time, platform, subprocess, requests

OUT = Path('/kaggle/working/qun_gpu_results')
OUT.mkdir(parents=True, exist_ok=True)

print('Python:', platform.python_version())
print('Platform:', platform.platform())

try:
    print(subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=20).stdout[:6000])
except Exception as e:
    print('nvidia-smi unavailable:', repr(e))

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU count:', torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {p.name}; VRAM={p.total_memory / 1024**3:.1f} GiB')
else:
    raise RuntimeError('GPU не подключён. В Kaggle открой Settings → Accelerator → GPU и перезапусти сессию.')

try:
    r = requests.get('https://huggingface.co', timeout=15)
    print('Hugging Face HTTP:', r.status_code)
except Exception as e:
    print('Internet check failed:', repr(e))

print('Output directory:', OUT)


In [ ]:
%pip install -q "diffusers>=0.32,<1.0" "transformers>=4.44,<5.0" accelerate safetensors


In [ ]:
import torch, json, time, zipfile
from diffusers import AutoPipelineForText2Image
from IPython.display import display

MODEL_ID = 'stabilityai/sd-turbo'
PROMPT = ('premium commercial product photography, elegant emerald and deep navy palette, '
          'soft natural studio lighting, refined minimal composition, realistic materials, '
          'high detail, clean background, no text, no watermark')
SEED = 4061109

started = time.time()
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
)
pipe = pipe.to('cuda')
generator = torch.Generator(device='cuda').manual_seed(SEED)
result = pipe(
    prompt=PROMPT,
    num_inference_steps=4,
    guidance_scale=0.0,
    width=512,
    height=512,
    generator=generator,
)
image = result.images[0]
image_path = OUT / 's40in_gpu_smoke_test.png'
image.save(image_path)

manifest = {
    'status': 'generated',
    'model': MODEL_ID,
    'seed': SEED,
    'prompt': PROMPT,
    'size': list(image.size),
    'elapsed_seconds': round(time.time() - started, 2),
    'gpu': torch.cuda.get_device_name(0),
    'created_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}
(OUT / 'manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
display(image)
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
zip_path = Path('/kaggle/working/qun_gpu_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file():
            z.write(p, p.relative_to(OUT.parent))
print('READY:', zip_path)
print('Files:', [str(p) for p in OUT.rglob('*') if p.is_file()])
print('Скачай qun_gpu_results.zip через панель Output/Files Kaggle до завершения сессии.')
